In [1]:
COPY_POWER = 10

In [2]:
import pandas as pd
import random

macbeth_df = pd.read_fwf('macbeth.txt')
macbeth_df.columns = ['text']

hamlet_df = pd.read_fwf('hamlet.txt')
hamlet_df.columns = ['text']

merchant_df = pd.read_fwf('merchantofvenice.txt')
merchant_df.columns = ['text']

romeo_df = pd.read_fwf('romeojuliet.txt')
romeo_df.columns = ['text']

# Combine the dataframes
df = pd.concat([macbeth_df, hamlet_df, merchant_df, romeo_df], ignore_index=True)

for i in range(COPY_POWER):
    df = pd.concat([df, df], ignore_index=True)
    
df['text'] = df['text'].astype('str')

In [3]:
# added to remove single letter text which messes up the railfence cipher
df = df[df['text'].str.len() > 2]

In [4]:
df

,text
0,THE TRAGEDY OF MACBETH
1,by William Shakespeare
2,Dramatis Personae
3,"DUNCAN, King of Scotland"
4,"MACBETH, Thane of Glamis and Cawdor, a general..."
...,...
13719547,"Some shall be pardon'd, and some punished;"
13719548,For never was a story of more woe
13719549,Than this of Juliet and her Romeo.
13719550,Exeunt omnes.


In [5]:
len(df.index)

13709312

In [6]:
def clean_str(input: str) -> str:
    # to lower case
    input = input.lower()

    # remove whitespace
    input = input.replace(" ", "")

    # remove punctuation
    input = input.replace(".", "")
    input = input.replace(",", "")
    input = input.replace(";", "")
    input = input.replace(":", "")
    input = input.replace("?", "")
    input = input.replace("!", "")
    input = input.replace("'", "")
    input = input.replace('"', "")
    input = input.replace("(", "")
    input = input.replace(")", "")
    input = input.replace("[", "")
    input = input.replace("]", "")
    input = input.replace("{", "")
    input = input.replace("}", "")
    input = input.replace("-", "")
    input = input.replace("_", "")

    return input

In [7]:
df['text'] = df['text'].transform(clean_str)
df

,text
0,thetragedyofmacbeth
1,bywilliamshakespeare
2,dramatispersonae
3,duncankingofscotland
4,macbeththaneofglamisandcawdorageneralinthekings
...,...
13719547,someshallbepardondandsomepunished
13719548,forneverwasastoryofmorewoe
13719549,thanthisofjulietandherromeo
13719550,exeuntomnes


In [8]:
# total_rows = len(df.index)
# for ind in df.index:
#     # to lower case
#     df['text'][ind] = df['text'][ind].lower()

#     # remove whitespace
#     df['text'][ind] = df['text'][ind].replace(" ", "")

#     # remove punctuation
#     df['text'][ind] = df['text'][ind].replace(".", "")
#     df['text'][ind] = df['text'][ind].replace(",", "")
#     df['text'][ind] = df['text'][ind].replace(";", "")
#     df['text'][ind] = df['text'][ind].replace(":", "")
#     df['text'][ind] = df['text'][ind].replace("?", "")
#     df['text'][ind] = df['text'][ind].replace("!", "")
#     df['text'][ind] = df['text'][ind].replace("'", "")
#     df['text'][ind] = df['text'][ind].replace('"', "")
#     df['text'][ind] = df['text'][ind].replace("(", "")
#     df['text'][ind] = df['text'][ind].replace(")", "")
#     df['text'][ind] = df['text'][ind].replace("[", "")
#     df['text'][ind] = df['text'][ind].replace("]", "")
#     df['text'][ind] = df['text'][ind].replace("{", "")
#     df['text'][ind] = df['text'][ind].replace("}", "")
#     df['text'][ind] = df['text'][ind].replace("-", "")
#     df['text'][ind] = df['text'][ind].replace("_", "")
#     print(f"{ind/total_rows*100:.5f}% done", end='\r')


# Split 0 -> Caesar Cipher

In [9]:
def caesar_encrypt(text, s=7):
    result = ""
    # transverse the plain text
    for i in range(len(text)):
        char = text[i]
        # Encrypt uppercase characters in plain text
        
        if (char.isupper()):
            result += chr((ord(char) + s-65) % 26 + 65)
        # Encrypt lowercase characters in plain text
        else:
            result += chr((ord(char) + s - 97) % 26 + 97)
    return result

In [10]:
caesar_df = pd.DataFrame(columns=['algorithm', 'encrypt'])

caesar_df['encrypt'] = [caesar_encrypt(text, random.randint(1, 26)) for text in df.text.values] 
caesar_df['algorithm'] = 'caesar'

In [11]:
caesar_df

,algorithm,encrypt
0,caesar,drodbkqoniypwkmlodr
1,caesar,nkiuxxumyetmwqebqmdq
2,caesar,cqzlzshrodqrnmzd
3,caesar,ribqobywbuctgqchzobr
4,caesar,kyazcrfrfylcmdejykgqylbayubmpyeclcpyjglrfcigleq
...,...,...
13709307,caesar,zvtlzohssilwhykvukhukzvtlwbupzolk
13709308,caesar,ajmizqzmrvnvnojmtjahjmzrjz
13709309,caesar,eslyestdzqufwtpelyospcczxpz
13709310,caesar,ohoexdywxoc


# Split 1 -> Affine Cipher

## Code for Affine Cipher

In [12]:
class Affine(object):
    def __init__(self):
        pass

    def egcd(self, a, b): 
        x,y, u,v = 0,1, 1,0
        while a != 0: 
            q, r = b//a, b%a 
            m, n = x-u*q, y-v*q 
            b,a, x,y, u,v = a,r, u,v, m,n 
        gcd = b 
        return gcd, x, y 

    def modinv(self, a, m): 
        gcd, x, y = self.egcd(a, m) 
        if gcd != 1: 
            return None # modular inverse does not exist 
        else: 
            return x % m 
    
    def encrypt(self, text, key): 
        #E = (a*x + b) % 26 
        t = text.upper()
        res = []
        for index, c in enumerate(t):
            if c == ' ':
                res.append(' ')
                continue

            enc = ((key[0]*(ord(c) - ord('A')) + key[1]) % 26) + (ord('A') if text[index].isupper() else ord('a'))
            res.append(chr(enc))
        return ''.join(res)


    def decrypt(self, cipher, key): 
        #D(E) = (a^-1 * (E - b)) % 26
        cipher = cipher.upper().replace(' ', '')
        return ''.join([ chr(((self.modinv(key[0], 26)*(ord(c) - ord('A') - key[1])) % 26) + ord('A')) for c in cipher ]) 
      
affine = Affine()

In [13]:
affine_df = pd.DataFrame(columns=['algorithm', 'encrypt'])
affine_df['encrypt'] = [affine.encrypt(text, (random.randint(1, 25), random.randint(1, 25))) for text in df.text.values]
affine_df['algorithm'] = 'affine'

In [14]:
affine_df

,algorithm,encrypt
0,affine,wkhwudjhgbripdfehwk
1,affine,aqsgqqgocwuoekwmkokk
2,affine,uklvlqjnexknbylx
3,affine,fprjrrdlrtnxxjntzrrf
4,affine,hjnlrvxvxjjrltvfjhztjjpnjbplrjvrjrrjfzjvxrdzjvt
...,...,...
13709307,affine,ewscevuddjclupnwhnuhnewsclihkevcn
13709308,affine,lzvjvhvvxjljlbzvdzltzvvxzv
13709309,affine,vjcpvjkuqhlwnkgvcpfjgttqogq
13709310,affine,eaecyugqyem


# Split 2 -> Vigenere Cipher

In [15]:
MIN_KEY_LEN = 3
MAX_KEY_LEN = 10
ALPHA = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

class Vigenere:
    def __init__(self, key=None):
        if key is None:
            key = self.generate_random_key()
        self.key = key

    def get_key(self):
        return self.key
    
    def set_key(self):
        key = self.generate_random_key()

    def generate_random_key(self) -> str:
        length = random.randint(MIN_KEY_LEN, MAX_KEY_LEN + 1)
        return ''.join(random.sample(ALPHA, length))

    def generate_keyword(self, n: int) -> str:
        return ((n // len(self.key) + 1) * self.key)[:n]

    def get_ord(self, letter):
        return ALPHA.index(letter.upper())
    
    def get_chr(self, num, uppercase=True):
        return ALPHA[num % 26] if uppercase else ALPHA[num % 26].lower()
    
    def encrypt(self, plaintext: str) -> str:
        keyword = self.generate_keyword(len(plaintext))
        return ''.join([self.get_chr(self.get_ord(ch) + self.get_ord(keyword[i]), ch.isupper())
                        if ch.upper() in ALPHA else ch for i, ch in enumerate(plaintext)])

    def decrypt(self, ciphertext: str) -> str:
        keyword = self.generate_keyword(len(ciphertext))
        return ''.join([self.get_chr(self.get_ord(ch) - self.get_ord(keyword[i]), ch.isupper())
                        if ch.upper() in ALPHA else ch for i, ch in enumerate(ciphertext)])

In [16]:
vigenere_test = Vigenere('TESTING')
plaintext = 'this is a test. Hello!'
ciphertext = vigenere_test.encrypt(plaintext)
print(plaintext)
print(ciphertext)

vigenere_test.decrypt(ciphertext)

this is a test. Hello!
mlal vy e mmfz. Zxtyu!


'this is a test. Hello!'

In [17]:
vigenere = Vigenere()

vigenere_df = pd.DataFrame(columns=['algorithm', 'encrypt'])

def vigenere_encrypt_with_random_key(vigenere, text):
    vigenere.set_key()
    return vigenere.encrypt(text)

vigenere_df['encrypt'] = [vigenere_encrypt_with_random_key(vigenere, text) for text in df.text.values]
vigenere_df['algorithm'] = 'vigenere'

vigenere_df

,algorithm,encrypt
0,vigenere,tfqtpmgcpymrmyobcfh
1,vigenere,bwiijxiyysfmkcepcmrc
2,vigenere,dpmmyfiqbepeolme
3,vigenere,dszcyzkgzgmrsaatjmnb
4,vigenere,myobcfhrtalqodslyyiqmnboaupopmgczepmlgztfqkgzgq
...,...,...
13709307,vigenere,smyeqtajxbcbappolpalpsmyengngehcp
13709308,vigenere,fmdnchepiaqmsrarwafkarcioc
13709309,vigenere,tfmnrtiqafhglgqtyzdfqrpamca
13709310,vigenere,evqulfokzeq


# Split 3 -> Substitution Cipher 

In [18]:
def substitution_encrypt(text):
    map = [chr(i+ord('a')) for i in range(26)]
    random.shuffle(map)
    for i in range(26):
        c = chr(i+ord('a'))
        text = text.replace(c, map[i])
    return text

In [19]:
substitution_df = pd.DataFrame(columns=['algorithm', 'encrypt'])
substitution_df['encrypt'] = [substitution_encrypt(text) for text in df.text.values] 
substitution_df['algorithm'] = 'substitution'

In [20]:
substitution_df

,algorithm,encrypt
0,substitution,hnahobdauchocbagahn
1,substitution,phrgoogahgdaragkaaqa
2,substitution,apeaecchdnpheben
3,substitution,fihamhlihdgckagmnmhf
4,substitution,aboldbfbfbldgjcabadablhobfhgabcdldabadlbfdkdlca
...,...,...
13709307,substitution,rlcxrcheegxjhbjltjhtjrlcxjethrcxj
13709308,substitution,ynahcucamdedeanawnyenacmnc
13709309,substitution,qfilqfgsmbebcgdqiljfdccmqdm
13709310,substitution,aaaohqkghak


# Split 4 -> Railfence Cipher

In [21]:
class RailFence(object):
    def __init__(self):
        pass

    def matrix(self, text, key): 
        try:
            rail = [['\n' for i in range(len(text))] for j in range(key)]
            dir_down = False
            row, col = 0, 0

            for i in range(len(text)):
                if (row == 0) or (row == key - 1):
                    dir_down = not dir_down
                rail[row][col] = text[i]
                col += 1
                if dir_down:
                    row += 1
                else:
                    row -= 1
        except:
            print("ERROR", text, key)

        return rail
    
    def encrypt(self, text, key): 

        rail=self.matrix(text,key)
        result = []
        for i in range(key):
            for j in range(len(text)):
                if rail[i][j] != '\n':
                    result.append(rail[i][j])
        return("" . join(result))

        
rail_fence = RailFence()

In [22]:
rail_fence_df = pd.DataFrame(columns=['algorithm', 'encrypt'])
rail_fence_df['encrypt'] = [rail_fence.encrypt(text, random.randint(2, len(text))) for text in df.text.values]
rail_fence_df['algorithm'] = 'rail_fence'


In [23]:
rail_fence_df

,algorithm,encrypt
0,rail_fence,thehttreabgceadmyfo
1,rail_fence,bywilleiraamesphsaek
2,rail_fence,dapormtsesneaira
3,rail_fence,dlutanonccdasnfkoign
4,rail_fence,mnaeecgrbaaerltoihdntwthahacendkenioanfsggislma
...,...,...
13709307,rail_fence,saeoprmpmedouebosnslndidhldnseaah
13709308,rail_fence,fyororofntmesovareserawewo
13709309,rail_fence,thanthioseomfojrurleihedtna
13709310,rail_fence,exesuenntmo


# Exporting to CSV

In [24]:
def export_df(df, filename):
    df.to_csv(filename, index=False)

# combine all the dataframes
combined_df = pd.concat([caesar_df, affine_df, vigenere_df, rail_fence_df, substitution_df])

# drop any duplicate rows
combined_df = combined_df.drop_duplicates()

# shuffle the data
combined_df = combined_df.sample(frac=1).reset_index(drop=True)

# export the combined dataframe
export_df(combined_df, 'dataset.csv')

In [25]:
combined_df

,algorithm,encrypt
0,substitution,lboiboeohhohncwibechhwwononibnnhonwla
1,substitution,vdqxvdevxdengadbdedenaeqvfadb
2,substitution,kldmektlltettehokmoktboitttmebihbdtttteltbemeh...
3,affine,gxsshsnhaakxdlpxyyatswtgvtfqppy
4,affine,jvfthllfpdpffnlhjbfvbnbphrfvblzlfzhvhvf
...,...,...
19523417,affine,iwmeqcscsewwmwke
19523418,affine,qcesscgaoccgqqygeqgegqkiyqsqc
19523419,substitution,ahhiaainmcnmagnenmgenafhgnabihignin
19523420,affine,hnzgutlotlyeilonzgkngtkzgykniglypt


# Ciphertext generation

In [26]:
text = "thekingsofscotland"

print("Original Text:", text)
print("Caesar Cipher:", caesar_encrypt(text, random.randint(2, 22)))
print("Affine Cipher:", affine.encrypt(text, (random.randint(2, 24), random.randint(2, 24))))
print("Vigenere Cipher:", vigenere.encrypt(text))
print("Rail Fence Cipher:", rail_fence.encrypt(text, random.randint(3, len(text))))
print("Substitution Cipher:", substitution_encrypt(text))


Original Text: thekingsofscotland
Caesar Cipher: vjgmkpiuqhueqvncpf
Affine Cipher: wkuayqwieiisewoqqg
Vigenere Cipher: tfqkgzgqafqoorxalp
Rail Fence Cipher: tshfceookstdiglnna
Substitution Cipher: ieaejcgdnxddnibeci
